# Paddy Disease Detection - MobileNetV2
## Lightweight Model for Mobile Deployment

This notebook uses MobileNetV2 for paddy disease classification.
MobileNetV2 is optimized for mobile and edge devices with fewer parameters.

## 1. Import Libraries

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import os
import pickle

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
from PIL import Image

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Metrics
from sklearn.metrics import classification_report, confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

2026-02-10 16:38:07.256459: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 16:38:07.336546: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-10 16:38:08.890735: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration

In [2]:
# Paths
BASE_DIR = '../data/raw'
TRAIN_DIR = os.path.join(BASE_DIR, 'train_images')
TEST_DIR = os.path.join(BASE_DIR, 'test_images')
MODEL_DIR = '../models'

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

# Model parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 1e-3
NUM_CLASSES = 10

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuration loaded!")

Configuration loaded!


## 3. Data Preprocessing & Augmentation

In [3]:
# Create data generators with MobileNetV2 preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

print("Data generators created with MobileNetV2 preprocessing!")

Data generators created with MobileNetV2 preprocessing!


In [4]:
# Create training generator
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)

# Create validation generator
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

# Get class names
class_indices = train_generator.class_indices
class_names = list(class_indices.keys())

print("\nClass Indices:")
for class_name, idx in class_indices.items():
    print(f"  {idx}: {class_name}")

Found 8330 images belonging to 10 classes.
Found 2077 images belonging to 10 classes.

Class Indices:
  0: bacterial_leaf_blight
  1: bacterial_leaf_streak
  2: bacterial_panicle_blight
  3: blast
  4: brown_spot
  5: dead_heart
  6: downy_mildew
  7: hispa
  8: normal
  9: tungro


## 4. Build MobileNetV2 Model

In [5]:
def build_mobilenet_model(num_classes=10, img_size=224):
    """
    Build MobileNetV2 model with transfer learning
    
    MobileNetV2 Features:
    - Inverted residual blocks with linear bottlenecks
    - Depthwise separable convolutions (fewer parameters)
    - Only ~3.4M parameters (vs 4M for EfficientNetB0)
    - Optimized for mobile/edge deployment
    """
    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    
    # Freeze base model layers initially
    base_model.trainable = False
    
    # Build model
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Build model
model, base_model = build_mobilenet_model(NUM_CLASSES, IMG_SIZE)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

I0000 00:00:1770721722.179264   10248 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,055,946 (11.66 MB)

 Trainable params: 793,866 (3.03 MB)

 Non-trainable params: 2,262,080 (8.63 MB)

## 5. Training

In [6]:
# Create callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'mobilenet_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured!")

Callbacks configured!


In [8]:
# Train the model (Phase 1: Frozen base)
print("=" * 50)
print("PHASE 1: Training with frozen base model")
print("=" * 50)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

# Save history
with open(os.path.join(MODEL_DIR, 'mobilenet_history_phase1.pkl'), 'wb') as f:
    pickle.dump(history.history, f)
print("Phase 1 history saved!")

PHASE 1: Training with frozen base model
Epoch 1/25
 25/261 ━━━━━━━━━━━━━━━━━━━━ 2:01 515ms/step - accuracy: 0.6118 - loss: 1.5639

KeyboardInterrupt: 

In [ ]:
# Fine-tuning: Unfreeze some layers
print("\n" + "=" * 50)
print("PHASE 2: Fine-tuning (unfreezing top layers)")
print("=" * 50)

# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Count trainable layers
trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Unfroze {trainable_layers} layers for fine-tuning")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history_fine = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

# Save fine-tuning history
with open(os.path.join(MODEL_DIR, 'mobilenet_history_phase2.pkl'), 'wb') as f:
    pickle.dump(history_fine.history, f)
print("Phase 2 history saved!")


PHASE 2: Fine-tuning (unfreezing top layers)
Unfroze 30 layers for fine-tuning
Epoch 1/15
 90/261 ━━━━━━━━━━━━━━━━━━━━ 1:34 550ms/step - accuracy: 0.5143 - loss: 1.9347

In [ ]:
# Save final model
model.save(os.path.join(MODEL_DIR, 'mobilenet_final.keras'))
print(f"Model saved to {MODEL_DIR}/mobilenet_final.keras")

## 6. Training Visualization

In [ ]:
# Load history if needed
try:
    h1 = history.history
except NameError:
    with open(os.path.join(MODEL_DIR, 'mobilenet_history_phase1.pkl'), 'rb') as f:
        h1 = pickle.load(f)

try:
    h2 = history_fine.history
except NameError:
    try:
        with open(os.path.join(MODEL_DIR, 'mobilenet_history_phase2.pkl'), 'rb') as f:
            h2 = pickle.load(f)
    except:
        h2 = None

In [ ]:
def plot_training_history(h1, h2=None):
    if h2:
        combined = {
            'accuracy': h1['accuracy'] + h2['accuracy'],
            'val_accuracy': h1['val_accuracy'] + h2['val_accuracy'],
            'loss': h1['loss'] + h2['loss'],
            'val_loss': h1['val_loss'] + h2['val_loss']
        }
        phase1_epochs = len(h1['accuracy'])
    else:
        combined = h1
        phase1_epochs = None
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy plot
    axes[0].plot(combined['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(combined['val_accuracy'], label='Val Accuracy', linewidth=2)
    if phase1_epochs:
        axes[0].axvline(x=phase1_epochs-1, color='r', linestyle='--', label='Fine-tuning Start')
    axes[0].set_title('MobileNetV2 - Model Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss plot
    axes[1].plot(combined['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(combined['val_loss'], label='Val Loss', linewidth=2)
    if phase1_epochs:
        axes[1].axvline(x=phase1_epochs-1, color='r', linestyle='--', label='Fine-tuning Start')
    axes[1].set_title('MobileNetV2 - Model Loss', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    best_val_acc = max(combined['val_accuracy'])
    best_epoch = combined['val_accuracy'].index(best_val_acc) + 1
    print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}% at Epoch {best_epoch}")

plot_training_history(h1, h2)

## 7. Model Evaluation

In [ ]:
# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)
print(f"\nMobileNetV2 Validation Accuracy: {val_accuracy * 100:.2f}%")
print(f"MobileNetV2 Validation Loss: {val_loss:.4f}")

In [ ]:
# Get predictions
val_generator.reset()
predictions = model.predict(val_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

# Classification report
print("\n" + "=" * 50)
print("MOBILENETV2 CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('MobileNetV2 - Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Inference

In [ ]:
def predict_disease(image_path, model, class_names, img_size=224):
    """
    Predict disease from a single image using MobileNetV2
    """
    img = Image.open(image_path)
    img = img.resize((img_size, img_size))
    img_array = np.array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)
    
    predictions = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]
    
    return {
        'class': class_names[predicted_class],
        'confidence': confidence,
        'all_predictions': dict(zip(class_names, predictions[0]))
    }

In [ ]:
# Test on random validation images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

val_generator.reset()
test_batch, test_labels = next(val_generator)
test_predictions = model.predict(test_batch, verbose=0)

for i in range(8):
    pred_idx = np.argmax(test_predictions[i])
    true_idx = np.argmax(test_labels[i])
    confidence = test_predictions[i][pred_idx]
    
    img = test_batch[i].copy()
    img = (img + 1) / 2
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    color = 'green' if pred_idx == true_idx else 'red'
    axes[i].set_title(f"Pred: {class_names[pred_idx]}\n"
                      f"True: {class_names[true_idx]}\n"
                      f"Conf: {confidence*100:.1f}%", 
                      color=color, fontsize=9)
    axes[i].axis('off')

plt.suptitle('MobileNetV2 Predictions (Green=Correct, Red=Incorrect)', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Model Summary

### MobileNetV2 Characteristics:
- **Parameters**: ~3.4M (lightweight)
- **Architecture**: Inverted residuals with linear bottlenecks
- **Best for**: Mobile apps, edge devices, real-time inference
- **Trade-off**: Slightly lower accuracy for much faster inference